# openCARP Monodomain Simulation Launcher

Pipeline di esecuzione:
1. **Configurazione** — percorsi paziente, parametri simulazione
2. **Validazione** — verifica esistenza di tutti i file di input
3. **Anteprima dati** — lettura `.vtx` e `.csv` dall'inferenza CDT
4. **Generazione script** — produce `run_mono_<case>.py` parametrizzato (carputils)
5. **Esecuzione** — `docker run` nel container openCARP con log in real-time
6. **Verifica output** — lista file prodotti dalla simulazione


In [1]:
import os, sys, stat, subprocess, uuid, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
sys.path.insert(0, "../src")
from path_config import set_path_mapping, get_path_mapping

path_mapping = get_path_mapping()

## 1 · Configurazione

In [23]:
# ── Identificativo paziente
CASE_NAME = "sb4301"

SIMULATION_CASE_DIR = os.path.join(".", CASE_NAME)
CASE_MESH_DIR = os.path.join(SIMULATION_CASE_DIR, f"{CASE_NAME}_meshes")
CASE_RN_DIR = os.path.join(SIMULATION_CASE_DIR,f"{CASE_NAME}_rootnodes_LBBB")
CASE_TRANSLATION_DIR = os.path.join(SIMULATION_CASE_DIR,"translation_files_LBBB")
RESULTS_CASE_DIR = os.path.join(path_mapping["results_path"], "personalisation_data", CASE_NAME)
SIMULATION_TAG = "Jun_2026_LBBB"
QRS_TRANSLATION_DIR = os.path.join(RESULTS_CASE_DIR, "qrs_stepFunction",SIMULATION_TAG, "for_translation_to_monodomain")
TWAVE_TRANSLATION_DIR = os.path.join(RESULTS_CASE_DIR,"twave_sf_IKs_GKs5_GKr0.5_tjca60_CL_810", SIMULATION_TAG + "_fixed_filter, for_translation_to_monodomain")
TRANSLATION_TAG = "best"
MONODOMAIN_RES = "fine500um"
SCRIPT_PATH = os.path.join(".","run_monodomain_torord_template_densesparse.py")
# ── INPUT: risultati inferenza CDT
ROOT_NODES_FILENAME = f"{CASE_NAME}_{MONODOMAIN_RES}_best_sel_root_nodes.csv"
RN_TIMES_FILENAME = f"{CASE_NAME}_{MONODOMAIN_RES}_best_sel_root_nodes_times.csv"
INIT_LATS_FILENAME  = f"{CASE_NAME}_{MONODOMAIN_RES}_nodefield_personalisation-biomarker_{TRANSLATION_TAG}_lat.dat"
GKS_SCALING_FILENAME = f"{CASE_NAME}_{MONODOMAIN_RES}_nodefield_personalisation-biomarker_{TRANSLATION_TAG}.adj"
ELECTRODES_FILENAME = f"{CASE_NAME}_electrodes_um"
# ── INPUT: mesh cardiaca (base path senza estensione) ────────────────────────
MESH_BASEPATH  = os.path.join(CASE_MESH_DIR, f"{CASE_NAME}_500um_tagged_fastendo_densesparse_v2")
# ── INPUT: file elettrodi (base path senza estensione) ───────────────────────
ELECTRODES_PATH = os.path.join(CASE_MESH_DIR, ELECTRODES_FILENAME)
RN_PATH = os.path.join(CASE_RN_DIR,ROOT_NODES_FILENAME)
RN_TIMES_PATH = os.path.join(CASE_RN_DIR,RN_TIMES_FILENAME)
LATS_PATH = os.path.join(CASE_TRANSLATION_DIR, INIT_LATS_FILENAME)
GKS_SCALING_PATH = os.path.join(CASE_TRANSLATION_DIR, GKS_SCALING_FILENAME)
# ── Libreria modello ionico Tomek 
TOMEK_FOLDER  = os.path.join(".","tomek-model")

# ── OUTPUT 
SIM_LABEL = "LBBB_translation_best_densesparse"
OUT_DIR   = os.path.join(SIMULATION_CASE_DIR, f"{CASE_NAME}_mono_torord_{SIM_LABEL}")

# ── Parametri simulazione 
TEND            = 500.0   # ms — durata totale
STIM_RADIUS_UM  = 2000    # µm — raggio ricerca nodi intorno a root node
STIM_DURATION   = 4.0     # ms
STIM_STRENGTH   = 53.0    # pA/pF
PREPACING_BEATS = 800
PREPACING_BCL   = 810.0   # ms

# ── Container openCARP 
OPENCARP_IMAGE = "docker.opencarp.org/opencarp/opencarp:latest"
NP             = 20  # numero processi MPI

print(f"Paziente  : {CASE_NAME}")
print(f"Output    : {OUT_DIR}")
print(f"Container : {OPENCARP_IMAGE}")

conductivity_myo = {
    'gil':0.3831,
    'git':0.2312,
    'gin':0.2524
}
conductivity_fast_dense = 0.73 #fastendo dense
conductivity_fast_sparse = 0.61
fastendo_scale_factor = 4.0

Paziente  : sb4301
Output    : ./sb4301/sb4301_mono_torord_LBBB_translation_best_densesparse
Container : docker.opencarp.org/opencarp/opencarp:latest


## 2 · Validazione file di input

In [21]:
'''
def check(path, label):
    p = Path(path)
    ok = p.exists()
    icon = '\u2713' if ok else '\u2717'
    size = f'  ({p.stat().st_size/1024:.0f} KB)' if ok and p.is_file() else ''
    print(f'  {icon}  {label:38s}  {p}{size}')
    return ok

print('── File di input ───────────────────────────────────────────────────────────')
ok_list = [
    check(VTX_FILE,                'Root nodes (.vtx)'),
    check(TIMES_FILE,              'Tempi attivazione (.csv)'),
    check(INIT_LATS,               'LAT iniziali pre-pacing (.dat)'),
    check(GKS_ADJ,                 'Gradiente GKs (.adj)'),
    check(str(MESH_BASE)+'.pts',   'Mesh (.pts)'),
    check(str(MESH_BASE)+'.elem',  'Mesh (.elem)'),
    check(str(MESH_BASE)+'.lon',   'Mesh fibre (.lon)'),
    check(str(ELECTRODES)+'.pts',  'Elettrodi (.pts)'),
    check(str(ELECTRODES)+'.elem', 'Elettrodi (.elem)'),
    check(TOMEK_LIB,               'Libreria Tomek (.so)'),
]
print()
if all(ok_list):
    print('\u2713  Tutti i file presenti.')
else:
    n_miss = sum(1 for x in ok_list if not x)
    print(f'\u2717  {n_miss} file mancanti. Aggiusta i percorsi prima di proseguire.')
'''

"\ndef check(path, label):\n    p = Path(path)\n    ok = p.exists()\n    icon = '✓' if ok else '✗'\n    size = f'  ({p.stat().st_size/1024:.0f} KB)' if ok and p.is_file() else ''\n    print(f'  {icon}  {label:38s}  {p}{size}')\n    return ok\n\nprint('── File di input ───────────────────────────────────────────────────────────')\nok_list = [\n    check(VTX_FILE,                'Root nodes (.vtx)'),\n    check(TIMES_FILE,              'Tempi attivazione (.csv)'),\n    check(INIT_LATS,               'LAT iniziali pre-pacing (.dat)'),\n    check(GKS_ADJ,                 'Gradiente GKs (.adj)'),\n    check(str(MESH_BASE)+'.pts',   'Mesh (.pts)'),\n    check(str(MESH_BASE)+'.elem',  'Mesh (.elem)'),\n    check(str(MESH_BASE)+'.lon',   'Mesh fibre (.lon)'),\n    check(str(ELECTRODES)+'.pts',  'Elettrodi (.pts)'),\n    check(str(ELECTRODES)+'.elem', 'Elettrodi (.elem)'),\n    check(TOMEK_LIB,               'Libreria Tomek (.so)'),\n]\nprint()\nif all(ok_list):\n    print('✓  Tutti i file pres

In [17]:
from pathlib import Path
GKS_COMPUTED_PATH = os.path.join(CASE_TRANSLATION_DIR, f"{CASE_NAME}_{MONODOMAIN_RES}_biomarker_GKs_scaled_{TRANSLATION_TAG}.adj")
GKs_ref_value = 0.0011 * 5.0
print(GKS_SCALING_PATH)
print(GKS_COMPUTED_PATH)
lines = Path(GKS_SCALING_PATH).read_text().splitlines()
header = []
data   = []
for line in lines:
    stripped = line.strip()
    if not stripped:
        continue
    parts = stripped.split()
    # Prima riga = numero nodi (intero)
    # Seconda riga = "intra" o "extra"
    if len(parts) == 1:
        header.append(stripped)
    elif len(parts) == 2:
        try:
            node_id = int(parts[0])
            value   = float(parts[1])
            data.append((node_id, value))
        except ValueError:
            # Es. "intra" / "extra"
            header.append(stripped)
    else:
        header.append(stripped)
scaled = []
for nid, factor in data:
    scaled_val = GKs_ref_value * factor
    scaled.append((nid, scaled_val))
fmt = f"{{}}  {{:.{4}f}}"
out = Path(GKS_COMPUTED_PATH).resolve()
out.parent.mkdir(parents=True, exist_ok=True)
lines = header + [fmt.format(nid, val) for nid, val in scaled]
out.write_text("\n".join(lines) + "\n")

./sb4301/translation_files_LBBB/sb4301_fine500um_nodefield_personalisation-biomarker_best.adj
./sb4301/translation_files_LBBB/sb4301_fine500um_biomarker_GKs_scaled_best.adj


33389864

## 3 · Anteprima dati di inferenza

In [18]:
nodes_raw = np.loadtxt(RN_PATH, dtype=int)
nodes = [nodes_raw.item()] if nodes_raw.ndim == 0 else list(nodes_raw)

print(nodes)
times_raw = np.loadtxt(RN_TIMES_PATH, dtype=float)
times = [times_raw.item()] if times_raw.ndim == 0 else list(times_raw)
print(times)

assert len(nodes) == len(times), (
    f'Mismatch: {len(nodes)} root nodes vs {len(times)} tempi')

print(f"Root nodes totali : {len(nodes)}")
print(f"\nPrime 10 righe:")
print(times)


[np.int64(1145269), np.int64(1738567), np.int64(577754), np.int64(1920212), np.int64(1938164)]
[np.float64(42.007), np.float64(36.3547), np.float64(27.9193), np.float64(33.1737), np.float64(29.289)]
Root nodes totali : 5

Prime 10 righe:
[np.float64(42.007), np.float64(36.3547), np.float64(27.9193), np.float64(33.1737), np.float64(29.289)]


## 4 · Generazione script carputils

Lo script viene scritto in `<PATIENT_DIR>/scripts/` con tutti i path assoluti del paziente già incorporati. Il container lo trova allo stesso path perché la cartella è montata con bind mount transparente.


In [24]:

sim_args = [
    '--casename',            CASE_NAME,
    '--meshname_path',   str(MESH_BASEPATH),
    '--rootnodes_path',        str(RN_PATH),
    '--times_path',      str(RN_TIMES_PATH),
    '--electrodes',      str(ELECTRODES_PATH),
    '--lat_path',       str(LATS_PATH),
    '--ionic_adj_path',         str(GKS_COMPUTED_PATH),
    '--conductivity_myo_gil', str(conductivity_myo["gil"]),
    '--conductivity_myo_git', str(conductivity_myo["git"]),
    '--conductivity_myo_gin', str(conductivity_myo["gin"]),
     '--conductivity_fast_dense', str(conductivity_fast_dense),
    '--conductivity_fast_sparse', str(conductivity_fast_sparse),   
    '--fast_scale', str(fastendo_scale_factor),
    '--inputdir',       str(SIMULATION_CASE_DIR),
    '--outdir',          str(OUT_DIR),
    '--tend',            str(TEND),
    '--prepacing_beats', str(PREPACING_BEATS),
    '--prepacing_cycle_length',   str(PREPACING_BCL),
    '--np', str(NP)
]

print('── Argomenti simulazione ───────────────────────────────────────────────────')
for i in range(0, len(sim_args), 2):
    print(f'  {sim_args[i]:25s}  {sim_args[i+1]}')

── Argomenti simulazione ───────────────────────────────────────────────────
  --casename                 sb4301
  --meshname_path            ./sb4301/sb4301_meshes/sb4301_500um_tagged_fastendo_densesparse_v2
  --rootnodes_path           ./sb4301/sb4301_rootnodes_LBBB/sb4301_fine500um_best_sel_root_nodes.csv
  --times_path               ./sb4301/sb4301_rootnodes_LBBB/sb4301_fine500um_best_sel_root_nodes_times.csv
  --electrodes               ./sb4301/sb4301_meshes/sb4301_electrodes_um
  --lat_path                 ./sb4301/translation_files_LBBB/sb4301_fine500um_nodefield_personalisation-biomarker_best_lat.dat
  --ionic_adj_path           ./sb4301/translation_files_LBBB/sb4301_fine500um_biomarker_GKs_scaled_best.adj
  --conductivity_myo_gil     0.3831
  --conductivity_myo_git     0.2312
  --conductivity_myo_gin     0.2524
  --conductivity_fast_dense  0.73
  --conductivity_fast_sparse  0.61
  --fast_scale               4.0
  --inputdir                 ./sb4301
  --outdir                 

## 5 · Esecuzione nel container openCARP

La funzione `run_in_opencarp()` costruisce il comando `docker run` con:
- bind mount trasparente di `PATIENT_DIR` e della cartella Tomek
- stesso UID/GID dell'utente host (nessun problema di permessi sull'output)
- log in streaming riga per riga


In [21]:
def run_in_opencarp(
    script: Path,
    args: list,
    image: str = OPENCARP_IMAGE,
    extra_volumes: dict = None,
    container_name: str = None,
) -> int:
    script = Path(script).resolve()
    
    # Volumi da montare (path identici host = container)
    vols = {
        str(script.parent):str(script.parent),
        str(Path(SIMULATION_CASE_DIR).resolve()):str(Path(SIMULATION_CASE_DIR).resolve()),
        str(Path(TOMEK_FOLDER).resolve()):str(Path(TOMEK_FOLDER).resolve()),
    }
    if extra_volumes:
        vols.update(extra_volumes)

    uid, gid = os.getuid(), os.getgid()
    cname = container_name or f'opencarp_{CASE_NAME}_{TRANSLATION_TAG}'

    docker_cmd = [
        'docker', 'run', '--rm',
        '--name', cname,
        '-u', f'{uid}:{gid}',
        '-e', 'PYTHONUNBUFFERED=1',
    ]
    for hp, cp in vols.items():
        docker_cmd += ['-v', f'{hp}:{cp}']
    docker_cmd += ['-w', str(script.parent)]
    docker_cmd += ['--shm-size=512m']
    docker_cmd += [image, 'python3', str(script)]
    docker_cmd += args

    print('── Comando docker ──────────────────────────────────────────────────────────')
    # Stampa leggibile: un argomento per riga
    print('docker run ... opencarp \\\n  python3', str(script), '\\')
    for i in range(0, len(args)-1, 2):
        cont = '\\' if i+2 < len(args) else ''
        print(f'  {args[i]} {args[i+1]} {cont}')
    print('─' * 78)
    print(f'Avvio: {datetime.now().strftime("%H:%M:%S")}')
    print()

    proc = subprocess.Popen(
        docker_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    #print('\n' + '─' * 78)
    ts = datetime.now().strftime('%H:%M:%S')
    status = '\u2713  Simulazione completata.' if proc.returncode == 0 \
             else f'\u2717  Errore — return code {proc.returncode}'
    print(f'[{ts}] {status}')
    return proc.returncode


### Avvia la simulazione

Esegui la cella seguente. I log di openCARP appaiono in tempo reale.


In [ ]:
rc = run_in_opencarp(
    script         = SCRIPT_PATH,
    args           = sim_args,
    image          = OPENCARP_IMAGE,
    container_name = f'opencarp_{CASE_NAME}',
)



## 6 · Stato container in corso

In [8]:
r = subprocess.run(
    ['docker', 'ps',
     '--filter', f'name=opencarp_{CASE_NAME}',
     '--format', 'table {{.Names}}\t{{.Status}}\t{{.RunningFor}}'],
    capture_output=True, text=True,
)
print(r.stdout or 'Nessun container openCARP attivo.')


NAMES     STATUS    CREATED



## 7 · Verifica file di output

In [7]:
if os.path.exists(OUT_DIR):
    files = sorted(OUT_DIR.rglob('*'))
    print(f'File in {OUT_DIR} ({len(files)} totali):\n')
    for f in files[:80]:
        rel  = f.relative_to(OUT_DIR)
        size = f'{f.stat().st_size/1024:.1f} KB' if f.is_file() else '(dir)'
        print(f'  {str(rel):<58s}  {size}')
    if len(files) > 80:
        print(f'  ... e altri {len(files)-80} file')
else:
    print('Directory di output non ancora creata.')


AttributeError: 'str' object has no attribute 'rglob'

## 8 · Interrompi simulazione (se necessario)

In [ ]:
# Decommentare per fermare il container:
# subprocess.run(['docker', 'stop', f'opencarp_{CASE_NAME}'])
# print(f'Container opencarp_{CASE_NAME} fermato.')
